<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**One row = one content item (`content_hash_id`) on one day (`report_date`)**, from
`fact_content_daily_performance`. This is the daily grain, not the 90-day-rolled-up grain the
starter CSV used — the warehouse gives me real daily history, so I can build my own prior-window
vs later-window comparison instead of relying on a single precomputed `trend_pct`.

**Time window for this contract:** I'm iterating on `month = '2026-03'`. The fact table ships
its own `month` column, which is almost certainly the partition key, so filtering on it directly
is both cheap and correct — and it keeps me away from `fact_content_daily_performance_sample`,
which is just June 2026 (the final month) and unsafe for developing label logic against.

**Which table(s):**
- `fact_content_daily_performance` — the daily signal: `gsc_impressions`, `gsc_clicks`,
  `gsc_avg_position`, plus GA4 and session-source columns I'm not using yet. Joined on
  `content_hash_id` + `client_hash_id` + `report_date`.
- `dim_content` — content-level context: `word_count`, `content_type`, `main_intent`. Ships as a
  single file (`dim_content.parquet`), not a folder — no glob needed. Joined on
  `content_hash_id`.
- I am NOT using `fact_content_query_90d` — its 90-day window overlaps my daily windows in ways
  I haven't aligned yet. Out of scope for W3.

**What I'd predict or rank (label/proxy):** whether a content item is showing the `answered_away`
pattern — impressions flat-or-rising while clicks fall meaningfully — versus `normal_decay`
(both fall together), computed from a prior 30-day window vs a later 30-day window I define
myself from daily rows.

**One thing deliberately excluded:** any FlyRank product-decision column (`health_score`,
`priority_score`, `action_type`, refresh flags) — these aren't shipped in the release, and if I
ever rebuild one myself it's a baseline to beat, never a feature or label.

In [10]:
# Section 1 code: connect to the warehouse, confirm both tables' real shapes and schemas.

import duckdb
from getpass import getpass

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_Token')
except Exception:
    hf_token = getpass('HF_Token: ')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
DIM  = f"read_parquet('{rel}/dim_content.parquet')"  # single file, confirmed — no glob

print("fact_content_daily_performance row count:")
print(con.sql(f"SELECT COUNT(*) FROM {FACT}").df())

print("\nfact_content_daily_performance schema:")
print(con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 0").df())

print("\ndim_content schema:")
print(con.sql(f"DESCRIBE SELECT * FROM {DIM} LIMIT 0").df())

# month is a real column on the fact table — filtering on it is likely partition-pruned
# and cheaper than a date range, so use it as the primary filter going forward.
print("\nMarch 2026 slice — row count + date span:")
print(con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT}
    WHERE month = '2026-03'
""").df())

fact_content_daily_performance row count:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   count_star()
0      78835655

fact_content_daily_performance schema:
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13  

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    n_rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

**Context (join/group only, never a feature):** `content_hash_id`, `client_hash_id`,
`report_date`, `url_hash_id`, `keyword_hash_id`. These codes carry no signal themselves.

**Features (must be knowable BEFORE the decision point — the end of the prior-30d window):**
1. `gsc_impressions_prior30` — summed GSC impressions over days -60 to -31 relative to the
   decision date.
2. `gsc_clicks_prior30` — summed GSC clicks, same window.
3. `gsc_avg_position_prior30` — mean position over the same prior window.
4. `word_count` (from `dim_content`) — static content property, doesn't change with the
   decision date, always knowable.
5. `content_type` / `main_intent` (from `dim_content`) — same, static content metadata.

**Label / proxy (never a feature):** `pattern_group`, computed from comparing
`gsc_impressions_prior30`/`gsc_clicks_prior30` against the last-30-day window's equivalents —
the LAST window defines the label, so nothing from it belongs in the feature list above. This is
the exact leak Week 2 found in `trend_direction`/`trend_pct`, applied here on purpose so it
never sneaks in again.

**Excluded, with why:**
- `health_score`, `priority_score`, `action_type` — not shipped in the release; if I ever
  rebuild one it's a baseline to beat, never a feature.
- `sessions_ai` and the per-assistant AI columns (`ai_chatgpt`, `ai_perplexity`, etc.) —
  genuinely interesting for a later AI-referral side note, but out of scope for this contract.
- `gsc_data_available` / `ga4_data_available` — used for the Section 3 availability check, but
  not fed to the model as a feature; they describe data quality, not page behavior.

In [11]:
# Section 2 code: confirm every column named above actually exists on the real schema
# pulled back in Section 1 — catches typos before Section 3 depends on them.

expected_fact_cols = {'report_date', 'client_hash_id', 'content_hash_id',
                       'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'month'}
expected_dim_cols  = {'content_hash_id', 'word_count', 'content_type', 'main_intent'}

fact_cols = set(con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 0").df()['column_name'])
dim_cols  = set(con.sql(f"DESCRIBE SELECT * FROM {DIM} LIMIT 0").df()['column_name'])

print("Missing from fact table (should be empty):", expected_fact_cols - fact_cols)
print("Missing from dim table (should be empty):", expected_dim_cols - dim_cols)

Missing from fact table (should be empty): set()
Missing from dim table (should be empty): set()


## 3. Verify it with queries (grain, counts, missing values, windows)

Three claims, each backed by a query below: the grain really is one row per content × day
(zero duplicates); the March slice's row count and date span match what Section 1 already
showed; and availability, filtered with `IS TRUE` (never `= FALSE`, since NULL is neither
TRUE nor FALSE and a careless filter silently drops those rows from the count).

Then: a 5-feature frame built from the same March month, and the deliberate leak experiment —
add one label-derived column on purpose, watch the score jump, delete it, keep the honest number.

In [12]:
# Query 1 — GRAIN: prove one row really is one content_id x one report_date.
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):")
print(grain_check)

# Query 2 — ROW COUNT + DATE SPAN, this slice specifically.
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT}
    WHERE month = '2026-03'
""").df()
print("\nMarch slice counts:")
print(counts)

# Query 3 — AVAILABILITY, filtered with IS TRUE. `= FALSE` or `NOT flag` silently drops
# NULL rows from the count — IS TRUE is the only safe filter.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_null_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_null_rows
    FROM {FACT}
    WHERE month = '2026-03'
""").df()
print("\nAvailability breakdown (note: NULL is neither TRUE nor FALSE):")
print(availability)

# Build prior-30 vs last-30 windows from daily rows myself, anchored inside March, so
# nothing here touches the same window that defines the label.
# Decision date = March 31; prior window = Feb 1-28; "last" window (label-defining,
# NOT a feature) = March 1-30.

feature_label_frame = con.sql(f"""
    WITH prior AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS gsc_impressions_prior30,
               SUM(gsc_clicks) AS gsc_clicks_prior30,
               AVG(gsc_avg_position) AS gsc_avg_position_prior30
        FROM {FACT}
        WHERE report_date >= DATE '2026-02-01' AND report_date < DATE '2026-03-01'
        GROUP BY content_hash_id, client_hash_id
    ),
    last AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS gsc_impressions_last30,
               SUM(gsc_clicks) AS gsc_clicks_last30
        FROM {FACT}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-03-31'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        p.content_hash_id, p.client_hash_id,
        p.gsc_impressions_prior30, p.gsc_clicks_prior30, p.gsc_avg_position_prior30,
        d.word_count, d.content_type, d.main_intent,
        l.gsc_impressions_last30, l.gsc_clicks_last30,
        CASE WHEN p.gsc_impressions_prior30 > 0
             THEN (l.gsc_impressions_last30 - p.gsc_impressions_prior30) * 1.0 / p.gsc_impressions_prior30 * 100
             ELSE NULL END AS impr_change_pct,
        CASE WHEN p.gsc_clicks_prior30 > 0
             THEN (l.gsc_clicks_last30 - p.gsc_clicks_prior30) * 1.0 / p.gsc_clicks_prior30 * 100
             ELSE NULL END AS click_change_pct
    FROM prior p
    JOIN last l USING (content_hash_id, client_hash_id)
    JOIN {DIM} d USING (content_hash_id)
    WHERE p.gsc_impressions_prior30 >= 50 AND p.gsc_clicks_prior30 >= 3
""").df()

def assign_pattern(row):
    if row['impr_change_pct'] is not None and row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] is not None and row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

feature_label_frame['pattern_group'] = feature_label_frame.apply(assign_pattern, axis=1)
print(feature_label_frame['pattern_group'].value_counts())
print(f"\nTotal rows in feature frame: {len(feature_label_frame):,}")

# Feature justification — one "knowable at the decision moment" line each:
# 1. gsc_impressions_prior30 — summed only from Feb rows, entirely before the decision date.
# 2. gsc_clicks_prior30 — same window, same reasoning.
# 3. gsc_avg_position_prior30 — averaged only over Feb, before the decision moment.
# 4. word_count — static content property; doesn't change day to day, knowable in advance.
# 5. content_type / main_intent — same, static metadata knowable in advance.


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import pandas as pd

df = feature_label_frame.dropna(subset=['gsc_avg_position_prior30']).copy()
y = (df['pattern_group'] == 'answered_away').astype(int)

honest_features = ['gsc_impressions_prior30', 'gsc_clicks_prior30', 'gsc_avg_position_prior30', 'word_count']
X_honest = pd.get_dummies(df[honest_features + ['content_type', 'main_intent']], dummy_na=True)

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
clf_honest = RandomForestClassifier(n_estimators=200, random_state=42)
clf_honest.fit(X_train, y_train)
honest_f1 = f1_score(y_test, clf_honest.predict(X_test))
print(f"HONEST F1 (answered_away class): {honest_f1:.3f}")

# --- THE TRAP, ON PURPOSE ---
# Sneak in click_change_pct — computed directly from the SAME last-30 window that
# defines the label. Deliberate, mirroring the trend_direction/trend_pct leak.
X_leaky = X_honest.copy()
X_leaky['click_change_pct_LEAKY'] = df['click_change_pct'].fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
clf_leaky = RandomForestClassifier(n_estimators=200, random_state=42)
clf_leaky.fit(X_train_l, y_train_l)
leaky_f1 = f1_score(y_test_l, clf_leaky.predict(X_test_l))
print(f"LEAKY F1 (with click_change_pct as a 'feature'): {leaky_f1:.3f}")
print(f"\nJump from leakage: {leaky_f1 - honest_f1:+.3f}")

# --- REMOVE IT, KEEP THE HONEST NUMBER ---
print(f"\nFinal, kept number for this lane going forward: honest F1 = {honest_f1:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be empty):
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, n]
Index: []


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


March slice counts:
    n_rows  n_content   min_date   max_date
0  9841378     331437 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability breakdown (note: NULL is neither TRUE nor FALSE):
   total_rows  gsc_available_rows  gsc_null_rows  ga4_available_rows  \
0     9841378             3611061              0              413966   

   ga4_null_rows  
0        3018741  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pattern_group
stable_other     18820
answered_away     6717
normal_decay      3979
Name: count, dtype: int64

Total rows in feature frame: 29,516
HONEST F1 (answered_away class): 0.135
LEAKY F1 (with click_change_pct as a 'feature'): 0.695

Jump from leakage: +0.560

Final, kept number for this lane going forward: honest F1 = 0.135


## 4. Data limits


**What this data can never tell you:**

* **No SERP feature confirms the mechanism.** Nothing in this dataset confirms an AI Overview,
  a featured snippet, or any other specific thing appeared on the results page —
  `pattern_group` is a behavioral proxy for click suppression, not a measured cause.
  Proxy-of-a-proxy, never causal.

* **History depth differs per client — confirmed, not just suspected.** Client row counts in
  the March slice range from 341 to 988,497 rows, a ~2,899x spread, and some clients' earliest
  row in the month starts well after March 1st (one client's data only begins March 19th). A
  flat global March window silently gives some clients full 31-day coverage and others a
  fraction of that. Not corrected in this notebook — a real follow-up would join
  `dim_clients.gsc_data_start` and either filter out clients with too little March history or
  build per-client windows instead of one global calendar month.

* **GA4/GSC zero-fill before a client's own data start.** Rows before a client's real data
  start are marked with `gsc_data_available` / `ga4_data_available` FALSE or NULL, not
  genuinely zero activity. The Section 3 availability query shows this split is real: GSC
  availability is clean (0 NULLs, purely TRUE/FALSE), but GA4 has 3,018,741 genuinely NULL rows
  out of 9,841,378 — a real "couldn't measure" gap, not just "measured zero." This contract
  doesn't yet filter it out, since I'm using GSC-only features this week.

* **Window overlap risk with the query table.** `fact_content_query_90d`'s fixed 90-day window
  overlaps my daily-window definitions and isn't used here at all, precisely to avoid that
  alignment problem before it's solved properly.

* **March is one month, not a full seasonal cycle.** A single mid-panel month can't rule out
  seasonality-driven swings that would look like `answered_away` but aren't — these numbers are
  specific to March 2026 and haven't been checked against another month yet.

* **Five honest features aren't enough signal yet.** Honest F1 on `answered_away` came out to
  0.135 — weak, and expected at this stage. This week's job was proving the contract and the
  leakage discipline, not shipping a working classifier; stronger features are a problem for
  the modeling weeks ahead.

In [14]:
# Quick check backing the "history depth differs per client" limitation named above —
# confirms the claim is real, not just asserted. Not a fix, just proof.

client_row_counts = con.sql(f"""
    SELECT client_hash_id, COUNT(*) AS n_rows,
           MIN(report_date) AS earliest_row, MAX(report_date) AS latest_row
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY client_hash_id
    ORDER BY n_rows ASC
""").df()

print("Row count per client, March slice — smallest 5 clients:")
print(client_row_counts.head())
print("\nRow count per client, March slice — largest 5 clients:")
print(client_row_counts.tail())
print(f"\nRange: smallest client has {client_row_counts['n_rows'].min():,} rows, "
      f"largest has {client_row_counts['n_rows'].max():,} rows — "
      f"a {client_row_counts['n_rows'].max() / client_row_counts['n_rows'].min():.1f}x spread.")


Row count per client, March slice — smallest 5 clients:
            client_hash_id  n_rows earliest_row latest_row
0  client_2c32078d69f2cbad     341   2026-03-01 2026-03-31
1  client_f6f0cdf26d03d7bd     520   2026-03-19 2026-03-31
2  client_e00b29e582949543    1216   2026-03-23 2026-03-31
3  client_810019792c9b8efc    1812   2026-03-20 2026-03-31
4  client_a1203ffecad62470    2325   2026-03-01 2026-03-31

Row count per client, March slice — largest 5 clients:
             client_hash_id  n_rows earliest_row latest_row
50  client_62f4a7e64f5e0096  756660   2026-03-01 2026-03-31
51  client_08a6a72ff48e62c0  851275   2026-03-01 2026-03-31
52  client_73cda7b4e4f265ea  869640   2026-03-01 2026-03-31
53  client_3ffa76342f366962  904847   2026-03-01 2026-03-31
54  client_625b6439094e23e4  988497   2026-03-01 2026-03-31

Range: smallest client has 341 rows, largest has 988,497 rows — a 2898.8x spread.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.